# Tutorial: SBSI inference

This notebook covers only model inference: loading a trained measurement flow and blending emulator, preparing their aligned inputs, and predicting shear response. The future simulation-based shear-likelihood interface is marked TODO.

You provide every training, validation, and inference catalogue. Model names such as V3 and V3b are only shortcuts to external model paths; they do not select data or change the workflow.

Training and tuning are command-line workflows, documented in the repository README. They are intentionally not mixed into this inference notebook.

## 1. Setup

In [ ]:
from pathlib import Path

from sbs_shear import (
    EmulatorPairingConfig,
    ModelPaths,
    ResponsePredictor,
    get_model,
    load_catalogue,
    load_emulator,
    prepare_forward_catalogue,
    predict_blend_response,
)

## 2. Define your catalogues

SBSI never chooses an inference catalogue for you. API methods accept a user-supplied path or pandas DataFrame.

The bundled example is the same truth-level input catalogue used by the BlendEMU tutorial. It contains `RA`, `DEC`, `redshift`, `r`, `Re`, `sersic_n`, `axis_ratio`, and `position_angle`, plus simulator provenance and shear columns. These are enough to prepare both response-model views. If new simulation or measurement catalogues are needed, `job_generate_catalogues.sh` shows how to call BlendEMU's CLI; SBSI does not implement rendering or measurement.

In [ ]:
EXAMPLES = Path("examples") if Path("examples").is_dir() else Path(".")
INPUT_CATALOGUE = EXAMPLES / "data/example_catalog.feather"
input_catalogue = load_catalogue(INPUT_CATALOGUE)
EMULATOR_RESPONSE_CATALOGUE = Path("/path/to/emulator_response.feather")

input_catalogue.head()

## 3. Load models

Use any model paths. `get_model("V3")` and `get_model("V3b")` are convenience path presets only.

In [ ]:
# Frozen release paths:
models = get_model("V3")

# An arbitrary model uses the identical workflow:
custom_models = ModelPaths(
    flow_checkpoints=(
        Path("/path/to/models/flow_s1.pt"),
        Path("/path/to/models/flow_s2.pt"),
    ),
    emulator_model=Path("/path/to/models/emulator.json"),
    emulator_metadata=Path("/path/to/models/emulator_metadata.json"),
)

OBSERVING_CONDITIONS = {
    "pixel_size": 0.2,
    "zero_point": 30.0,
    "psf_fwhm": 0.73,
    "moffat_beta": 2.224,
    "pixel_rms": 0.312,
}

models

## 4. Predict shear response

`ResponsePredictor.load` loads the flow metadata and reads its training domain. Prediction then evaluates the full ensemble with common random numbers and combines

$$R_{\mathrm{model}} = R_{\mathrm{flow}} + R_{\mathrm{blend}}.$$

The two response models need different views of the same input scene. The flow uses one row per primary with intrinsic orientation and aggregate crowding features. The emulator uses one row per accepted primary-neighbour pair. SBSI derives and aligns both views; no measured-image catalogue or second flow catalogue is needed for this response calculation.

In [ ]:
# Load the two models. Use a GPU node for production flow prediction.
# emulator = load_emulator(models, conditions=OBSERVING_CONDITIONS, device="cpu")
# predictor = ResponsePredictor.load(models, device="cuda")
# print("Condition features:", predictor.condition_features)
# print("Measured targets:  ", predictor.target_features)
# print("Training domain:   ", predictor.domain)

As in the BlendEMU tutorial, start from the supplied input catalogue. `prepare_forward_catalogue` returns two public, inspectable views: `flow_inputs` (one row per retained primary) and `emulator_pairs` (possibly several rows per primary). Keeping them separate avoids weighting the flow by neighbour multiplicity. For multiple fields, pass `group_column` so neighbours are never paired across fields.

In [ ]:
# pairing = EmulatorPairingConfig.from_emulator(emulator)
# prepared = prepare_forward_catalogue(input_catalogue, config=pairing)
# prepared.emulator_pairs[["primary_row", "secondary_row", "distance"]].head()

# The pair-level predictions are summed into a Series keyed by primary_row.
# r_blend = predict_blend_response(emulator, prepared.emulator_pairs)

# prediction = predictor.predict(
#     prepared.flow_inputs,
#     blend_response=r_blend,
#     shear=0.02,
#     n_samples=64,
# )
# prediction.summary()

You can also reuse precomputed emulator output. Pass a separate response catalogue keyed by `(case, input_index)`:

```python
prediction = predictor.predict(
    prepared.flow_inputs,
    blend_catalogue=EMULATOR_RESPONSE_CATALOGUE,
    blend_response="R_blend",
)
```

If `R_blend` is already an aligned column in `prepared.flow_inputs`, `predictor.predict(prepared.flow_inputs)` is sufficient. Alignment must be established by stable object keys or by construction; do not assume two unrelated tables have the same row order.

The result exposes `prediction.flow`, `prediction.blend`, and `prediction.total` per object. `prediction.total_mean` is the catalogue mean response. `R_flow` alone is only self-response and is never the complete model.

For a matched simulation response, SBSI uses

$$m = R_{\mathrm{sim}} / R_{\mathrm{model}} - 1.$$

```python
m = prediction.multiplicative_bias(simulation_response)
print(f"m = {100 * m:+.3f}%")
```

## 5. Simulation-based shear inference — TODO

The final goal is catalogue-level shear inference using the measurement flow and neighbour emulator in one simulation-based likelihood. That likelihood has not yet been worked out and validated, so this tutorial intentionally provides no executable recipe.

The implementation must first define the joint likelihood, intrinsic population prior, latent neighbour-scene marginalization, detection and selection normalization, ensemble uncertainty, and simulation-based coverage tests. The existing posterior-grid code is a research prototype, not the completed Part 3 API.

## Minimal recipe

```python
models = ModelPaths(
    flow_checkpoints=(...),
    emulator_model=...,
    emulator_metadata=...,
)
emulator = load_emulator(models, conditions=my_conditions)
predictor = ResponsePredictor.load(models, device="cuda")
pairing = EmulatorPairingConfig.from_emulator(emulator)
prepared = prepare_forward_catalogue(my_input_catalogue, config=pairing)
r_blend = predict_blend_response(emulator, prepared.emulator_pairs)
prediction = predictor.predict(
    prepared.flow_inputs,
    blend_response=r_blend,
)
```

The model paths and catalogues can change; the SBSI workflow does not.